# Logistic Regression Assumptions — Cybersecurity Solution

**Short name (GitHub):** `LogReg_Cyber`  
Work the skeleton first. Numbers use scikit-learn 1.x, `penalty=None`, `solver='lbfgs'`, `random_state=0` (model) and `random_state=6` (imbalance). Solver drift of a few points is normal.


## Inline cheat-sheet

| Item | This file |
|------|-----------|
| Rows / positivity | 720 events, PHISH=243, BENIGN=477, rate ≈ 0.338 |
| Independence | unique `event_id` — True |
| 10-EPV | 243/10 = 24.3 |
| Outlier cut | `redirect_count` 99th pct ≈ 6, 9 rows dropped |
| Collinear pair | `url_length` ~ `path_length` (r ≈ 0.97) |
| Weak logit feature | `request_hour` |
| Core 5-feature test (rs=0) | acc≈0.81 · prec≈0.77 · rec≈0.65 · F1≈0.71 · AUC≈0.87 |
| CM @0.50 | TN 123, FP 15, FN 27, TP 51 |
| CM @0.25 | FN 7, FP 46 |
| CM @0.75 | FN 58, FP 3 |


## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score,
)
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

%matplotlib inline
sns.set_style("whitegrid")
np.set_printoptions(precision=4, suppress=True)
print("libraries ready")


## 1. Load and encode

In [ ]:
df = pd.read_csv("data/cyber_phishing.csv")
df["phishing"] = df["phishing"].map({"PHISH": 1, "BENIGN": 0}).astype(int)
print(df.head())
print(df.phishing.value_counts())
# BENIGN=477, PHISH=243. Positivity ≈ 0.3375


## 2. Assumptions I

In [ ]:
print(df.phishing.value_counts())
print("n classes:", df.phishing.nunique())


In [ ]:
unique_ids = df.event_id.nunique() == df.event_id.count()
print(unique_ids)  # True


In [ ]:
max_features = min(df.phishing.value_counts()) / 10
print(max_features)  # 24.3


In [ ]:
all_features = [
    "url_length", "path_length", "num_dots", "num_hyphens", "num_subdomains",
    "has_ip", "has_https", "has_at", "digit_ratio", "redirect_count", "request_hour",
]
cont_features = [
    "url_length", "path_length", "num_dots", "num_hyphens", "num_subdomains",
    "digit_ratio", "redirect_count", "request_hour",
]
plt.figure(figsize=(10, 4.2))
sns.boxplot(data=np.log(df[cont_features] + 0.01).apply(zscore))
plt.xticks(rotation=40, ha="right")
plt.title("Log-z boxplot — redirect_count has the long upper tail")
plt.tight_layout(); plt.show()


In [ ]:
q_hi = df["redirect_count"].quantile(0.99)
df_filtered = df[df["redirect_count"] < q_hi].copy()
print("q99 =", q_hi, " kept =", len(df_filtered), " dropped =", len(df) - len(df_filtered))


In [ ]:
plt.figure(figsize=(10, 4.2))
sns.boxplot(data=np.log(df_filtered[cont_features] + 0.01).apply(zscore))
plt.xticks(rotation=40, ha="right")
plt.title("After 99th-percentile filter on redirect_count")
plt.tight_layout(); plt.show()


## 3. Assumptions II

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
sns.regplot(x="url_length", y="phishing", data=df, logistic=True, ax=axes[0],
            scatter_kws={"alpha": 0.22, "s": 14})
axes[0].set_title("url_length — sigmoid-like")
sns.regplot(x="request_hour", y="phishing", data=df, logistic=True, ax=axes[1],
            scatter_kws={"alpha": 0.22, "s": 14})
axes[1].set_title("request_hour — flat / weak")
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df[all_features].corr(), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, annot_kws={"size": 7})
plt.title("Feature correlations")
plt.tight_layout(); plt.show()
correlated_pair = ["url_length", "path_length"]  # r ≈ 0.97
print("correlated_pair:", correlated_pair)


## 4. scikit-learn

In [ ]:
core = ["url_length", "num_subdomains", "has_ip", "has_https", "digit_ratio"]
outcome = "phishing"
x_train, x_test, y_train, y_test = train_test_split(
    df[core], df[outcome], random_state=0, test_size=0.3
)
log_reg = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000, solver="lbfgs")
print(log_reg.get_params())


In [ ]:
log_reg.fit(x_train, y_train)
coefficients = log_reg.coef_
intercept = log_reg.intercept_
print("coefficients:", coefficients)
print("intercept:", intercept)
# Typical: url_length +, num_subdomains +, has_ip +, has_https −, digit_ratio +


In [ ]:
y_pred = log_reg.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"accuracy\t{accuracy:.4f}")
print(f"precision\t{precision:.4f}")
print(f"recall   \t{recall:.4f}")
print(f"f1       \t{f1:.4f}")
# acc ≈ 0.806  prec ≈ 0.773  rec ≈ 0.654  f1 ≈ 0.708
# Accuracy looks fine because the majority class is benign. Catch-rate is the SOC number.


In [ ]:
test_conf_matrix = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["actual benign", "actual phish"],
    columns=["predicted benign", "predicted phish"],
)
print(test_conf_matrix)
# TN 123, FP 15, FN 27, TP 51  — 27 missed phish at the default cut


## 5. Thresholds

In [ ]:
y_pred_prob = log_reg.predict_proba(x_test)
y_pred_class = (y_pred_prob[:, 1] > 0.5) * 1.0
diff = np.array_equal(y_pred_class, y_pred)
print("same as predict()?", diff)


In [ ]:
print("CM 50%"); print(confusion_matrix(y_test, y_pred_class))
print("CM 25%"); print(confusion_matrix(y_test, (y_pred_prob[:, 1] > 0.25) * 1.0))
print("CM 75%"); print(confusion_matrix(y_test, (y_pred_prob[:, 1] > 0.75) * 1.0))
# 25%: FN 7  FP 46  — more tickets, far fewer misses
# 50%: FN 27 FP 15
# 75%: FN 58 FP 3   — too conservative for a mail gateway


In [ ]:
thresh = np.linspace(0, 1, 100)
false_negatives = []
for t in thresh:
    cm = confusion_matrix(y_test, (y_pred_prob[:, 1] > t) * 1.0)
    false_negatives.append(cm[1, 0])
thresh_choice = thresh[np.argmax(np.array(false_negatives) >= 8)]
print("thresh_choice =", thresh_choice)
print("FN at that t =", false_negatives[int(np.argmax(np.array(false_negatives) >= 8))])


## 6. ROC / AUC

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob[:, 1])
plt.figure(figsize=(6.2, 5.6))
plt.plot(fpr, tpr, color="darkorange", label="ROC curve")
idx = list(range(len(thresholds)))[1::4]
for i in idx:
    plt.text(fpr[i], tpr[i], f"{thresholds[i]:.2f}", fontsize=7)
clf = DummyClassifier(strategy="most_frequent", random_state=0)
clf.fit(x_train, y_train)
fpr_d, tpr_d, _ = roc_curve(y_test, clf.predict_proba(x_test)[:, 1])
auc_d = roc_auc_score(y_test, clf.predict_proba(x_test)[:, 1])
plt.plot(fpr_d, tpr_d, color="navy", ls="--", label=f"Dummy most-frequent (AUC={auc_d:.2f})")
plt.xlabel("False Positive Rate (benign flagged)")
plt.ylabel("True Positive Rate (phish caught)")
plt.title("ROC — 5-feature phishing logit")
plt.grid(True, alpha=0.4); plt.legend(loc="lower right")
plt.show()


In [ ]:
roc_auc = roc_auc_score(y_test, y_pred_prob[:, 1])
print("ROC AUC:", roc_auc)  # ≈ 0.872


## 7. Class imbalance

In [ ]:
x_train_u, x_test_u, y_train_u, y_test_u = train_test_split(
    df[core], df[outcome], random_state=6, test_size=0.3
)
print("unstrat train pos", float(y_train_u.mean()), "test pos", float(y_test_u.mean()))
x_train_str, x_test_str, y_train_str, y_test_str = train_test_split(
    df[core], df[outcome], random_state=6, test_size=0.3, stratify=df[outcome]
)
print("strat   train pos", float(y_train_str.mean()), "test pos", float(y_test_str.mean()))


In [ ]:
str_train_positivity_rate = float(y_train_str.mean())
str_test_positivity_rate = float(y_test_str.mean())
print(str_train_positivity_rate, str_test_positivity_rate)


In [ ]:
log_reg.fit(x_train_str, y_train_str)
y_pred_s = log_reg.predict(x_test_str)
recall_str = recall_score(y_test_str, y_pred_s)
accuracy_str = accuracy_score(y_test_str, y_pred_s)
print("stratified recall, acc:", recall_str, accuracy_str)
log_reg.fit(x_train_u, y_train_u)
print("unstrat (rs=6) recall, acc:",
      recall_score(y_test_u, log_reg.predict(x_test_u)),
      accuracy_score(y_test_u, log_reg.predict(x_test_u)))


In [ ]:
log_reg_bal = LogisticRegression(
    penalty=None, fit_intercept=True, max_iter=4000, solver="lbfgs",
    class_weight="balanced",
)
log_reg_bal.fit(x_train_u, y_train_u)
y_pred_b = log_reg_bal.predict(x_test_u)
recall_bal = recall_score(y_test_u, y_pred_b)
accuracy_bal = accuracy_score(y_test_u, y_pred_b)
print("balanced recall, acc:", recall_bal, accuracy_bal)
# Balanced weights usually lift catch-rate and give back a point or two of accuracy.


## 8. Alternate code

In [ ]:
try:
    import statsmodels.api as sm
    sm_fit = sm.Logit(y_train, sm.add_constant(x_train)).fit(disp=False)
    print(sm_fit.summary())
except Exception as exc:
    print("statsmodels unavailable or failed:", exc)
    print("sklearn coef", coefficients, "intercept", intercept)


In [ ]:
pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000),
)
pipe.fit(x_train, y_train)
print("scaled-pipeline acc", accuracy_score(y_test, pipe.predict(x_test)))
print("scaled-pipeline rec", recall_score(y_test, pipe.predict(x_test)))


In [ ]:
def predict_at(proba, t=0.5):
    return (np.asarray(proba) >= t).astype(int)

for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    pred = predict_at(y_pred_prob[:, 1], t)
    cm = confusion_matrix(y_test, pred)
    print(f"t={t:.1f}  FN={cm[1,0]:2d}  FP={cm[0,1]:2d}  rec={recall_score(y_test, pred):.3f}")


## 9. More practice

In [ ]:
extra = core + ["has_at", "num_dots"]
Xtr_e, Xte_e, ytr_e, yte_e = train_test_split(
    df[extra], df[outcome], random_state=0, test_size=0.3
)
m_e = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000)
m_e.fit(Xtr_e, ytr_e)
pe = m_e.predict_proba(Xte_e)[:, 1]
print("extra recall", recall_score(yte_e, m_e.predict(Xte_e)))
print("extra AUC   ", roc_auc_score(yte_e, pe))
print("core  AUC   ", roc_auc)


In [ ]:
logins = pd.read_csv("data/cyber_logins.csv")
print(logins.head())
print("positivity", logins.brute_force.mean())
print("max features (10-EPV)", logins.brute_force.value_counts().min() / 10)
print("bytes vs packets r =", logins[["bytes_sent", "packets"]].corr().iloc[0, 1])
Lx = logins[["failed_logins", "src_unique_ua", "geo_rare", "off_hours"]]
Ly = logins["brute_force"]
Ltr, Lte, lytr, lyte = train_test_split(Lx, Ly, random_state=0, test_size=0.3, stratify=Ly)
m_l = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000)
m_l.fit(Ltr, lytr)
print("coef", m_l.coef_, "intercept", m_l.intercept_)
lp = m_l.predict_proba(Lte)[:, 1]
print("login recall", recall_score(lyte, m_l.predict(Lte)), "auc", roc_auc_score(lyte, lp))
for t in np.linspace(0.15, 0.7, 12):
    cm = confusion_matrix(lyte, (lp >= t).astype(int))
    print(f"t={t:.2f} FN={cm[1,0]:2d} FP={cm[0,1]:2d}")


In [ ]:
gateway = (
    "Mail / web gateway: prevalence of phish is low in raw traffic, a miss reaches a human. "
    "Drive the threshold down and quote catch-rate plus tickets-per-hour, not accuracy."
)
war_room = (
    "IR war-room: analysts are already looking at a enriched case. A higher threshold is fine — "
    "precision matters because every extra lead burns an investigator."
)
board = (
    "Board pack: one AUC, one catch-rate at the agreed SOP cut, and the ticket volume that cut implies. "
    "No coefficient tables."
)
print(gateway); print(war_room); print(board)


## 10. Simulation

In [ ]:
# --- editable parameters ---
N = 720
N_REPS = 20
NOISE = 0.00
T = 0.50
TEST_SIZE = 0.30
SEED = 0
# ---------------------------
rng = np.random.default_rng(SEED)
rows_def, rows_bal = [], []
pool_X = df[core].to_numpy()
pool_y = df[outcome].to_numpy()
for r in range(N_REPS):
    idx = rng.choice(len(df), size=N, replace=(N > len(df)))
    X, y = pool_X[idx], pool_y[idx]
    try:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=r, stratify=y)
    except ValueError:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=r)
    if NOISE > 0:
        flip = rng.random(len(ytr)) < NOISE
        ytr = ytr.copy(); ytr[flip] = 1 - ytr[flip]
    for tag, cw in (("default", None), ("balanced", "balanced")):
        m = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000, class_weight=cw)
        m.fit(Xtr, ytr)
        proba = m.predict_proba(Xte)[:, 1]
        pred = (proba >= T).astype(int)
        rec = dict(
            recall=recall_score(yte, pred, zero_division=0),
            accuracy=accuracy_score(yte, pred),
            auc=roc_auc_score(yte, proba) if len(np.unique(yte)) == 2 else np.nan,
        )
        (rows_def if tag == "default" else rows_bal).append(rec)
sim_def = pd.DataFrame(rows_def); sim_bal = pd.DataFrame(rows_bal)
print("DEFAULT\n", sim_def.agg(["mean", "std"]).round(3))
print("BALANCED\n", sim_bal.agg(["mean", "std"]).round(3))
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, col in zip(axes, ["recall", "accuracy", "auc"]):
    ax.boxplot([sim_def[col].dropna(), sim_bal[col].dropna()], labels=["default", "balanced"])
    ax.set_title(col); ax.set_ylim(0.35, 1.02)
fig.suptitle(f"N={N}  T={T}  noise={NOISE}  reps={N_REPS}", y=1.03)
plt.tight_layout(); plt.show()


## 11. Audience rewrite

In [ ]:
expert = (
    "On the rs=0 hold-out (n=216) an unpenalized 5-feature logit attains AUC 0.87. "
    "Operating at t=0.25 rather than 0.50 moves the CM from (FN=27, FP=15) to (FN=7, FP=46). "
    "path_length was dropped (r≈0.97 with url_length). request_hour fails the logit-linearity screen. "
    "has_https enters with a negative coefficient, matching the TLS base rate on legitimate hosts."
)
technician = (
    "Encode PHISH/BENIGN as 1/0. Fit LogisticRegression(penalty=None) on url_length, "
    "num_subdomains, has_ip, has_https, digit_ratio. Score with predict_proba[:,1]. "
    "If the SOP is 'miss at most ~8 phish on this test list', sit near t=0.25 and expect "
    "a larger ticket queue. Re-check event_id uniqueness after the redirect_count 99th-pct filter."
)
executive = (
    "A five-signal URL score separates phishing from benign well enough to use as a first filter "
    "(AUC 0.87). The default 50% cut still misses about one in three phish on the test list. "
    "Dropping the cut to 25% catches most of them and roughly triples extra tickets. "
    "That volume-versus-miss trade is a CISO policy choice, not a model default."
)
nonspecialist = (
    "Some links look odd — very long, an IP address instead of a name, lots of digits, no lock icon. "
    "The model turns those clues into a 0–100 risk score. Using a cautious cut-off, it catches "
    "almost every bad link in our practice file and sends extra harmless ones to a human reviewer. "
    "That extra review is the price of missing fewer scams."
)
print("EXPERT\n", expert)
print("TECHNICIAN\n", technician)
print("EXECUTIVE\n", executive)
print("NONSPECIALIST\n", nonspecialist)


## 12. Applications

In [ ]:
applications = [
    "1. Phishing / malicious-URL scoring on a mail or web gateway (this notebook).",
    "2. Account-takeover / brute-force login detection from session features.",
    "3. Malware vs benign on a short static-feature scorecard (imports, entropy, size).",
    "4. SIEM alert triage — promote / demote a rule hit before it pages an analyst.",
    "5. DLP / data-exfil first-pass on destination rarity + bytes + off-hours.",
    "6. Vulnerability prioritization (exploited-in-the-wild vs not) from CVSS-like fields.",
    "7. Business-email-compromise flag from reply-to mismatch + urgency lexicon counts.",
    "8. Insider-risk screen from badge + print + USB counts (with heavy human review).",
    "9. Bot vs human on a login page (pointer cadence proxies, not raw packets).",
    "10. Cloud misconfiguration 'will this public bucket be abused' using exposure flags.",
]
not_appropriate = [
    "1. Multi-family malware taxonomy (ransomware / stealer / RAT) — not binary.",
    "2. Repeated beacons from the same host or campaign — independence is broken.",
    "3. APT detection with 12 confirmed incidents and 40 features — EPV collapse.",
    "4. Raw PCAP / payload bytes as columns — use a representation model first.",
    "5. Causal claim that 'disabling HTTPS caused phishing' — this is a risk score.",
    "6. Streaming concept drift with no periodic refit — coefficients go stale.",
    "7. Highly non-linear command-and-control patterns — trees / sequence models.",
    "8. Time-to-compromise or dwell-time — that is survival, not a 0/1 logit.",
]
for row in applications: print(row)
print("--- not appropriate ---")
for row in not_appropriate: print(row)


## 13. Saved figures

`logreg_cyber_flowchart.png`, `logreg_cyber_boxplot.png`, `logreg_cyber_boxplot_filtered.png`,
`logreg_cyber_logit_curves.png`, `logreg_cyber_corr_heatmap.png`, `logreg_cyber_proba_hist.png`,
`logreg_cyber_roc.png`, `logreg_cyber_threshold_sweep.png`, `logreg_cyber_simulation.png`.
